In [1]:
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import os

In [2]:
# Download VADER lexicon if not already downloaded
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Vishwa\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [3]:
df = pd.read_csv("preprocessed_tweets.csv")

In [4]:
# Make sure column exists
if 'Content' not in df.columns:
    raise ValueError("Expected column 'Content' not found in CSV.")

In [5]:
# Initialize VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Apply VADER to each tweet
df['sentiment'] = df['Content'].astype(str).apply(sia.polarity_scores)

# Split compound, pos, neu, neg into separate columns
df['compound'] = df['sentiment'].apply(lambda x: x['compound'])
df['positive'] = df['sentiment'].apply(lambda x: x['pos'])
df['neutral'] = df['sentiment'].apply(lambda x: x['neu'])
df['negative'] = df['sentiment'].apply(lambda x: x['neg'])

# Optional: Drop the nested 'sentiment' dict column
df.drop(columns=['sentiment'], inplace=True)

In [6]:
# Save to CSV
OUTPUT_PATH = os.path.join('sentiment_labeled_tweets.csv')
def save_sentiment_labeled_to_csv(new_df, file_path=OUTPUT_PATH):
    # Check if the labeled CSV already exists
    if os.path.exists(file_path):
        existing_df = pd.read_csv(file_path)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined_df = new_df

    
    # Drop duplicates based on 'Tweet ID'
    if 'Tweet ID' in combined_df.columns:
        combined_df = combined_df.drop_duplicates(subset=['Tweet ID'], keep='last')

    combined_df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"[INFO] Sentiment-labeled data saved to: {file_path} | Total rows: {len(combined_df)}")

# Usage after your sentiment labeling step
save_sentiment_labeled_to_csv(df)

[INFO] Sentiment-labeled data saved to: sentiment_labeled_tweets.csv | Total rows: 1297


In [7]:
df.columns

Index(['Name', 'Handle', 'Timestamp', 'Verified', 'Content', 'Comments',
       'Retweets', 'Likes', 'Analytics', 'Tags', 'Mentions', 'Emojis',
       'Profile Image', 'Tweet Link', 'Tweet ID', 'cleaned_text', 'tokens',
       'compound', 'positive', 'neutral', 'negative'],
      dtype='object')

In [8]:
# Load your labeled dataset
df = pd.read_csv('sentiment_labeled_tweets.csv')

In [9]:
# Step 1: Create target sentiment labels
def label_sentiment(compound):
    if compound >= 0.05:
        return 'Positive'
    elif compound <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_label'] = df['compound'].apply(label_sentiment)


# Step 2: Features (cleaned_text) and target (sentiment_label)
X = df['cleaned_text'].astype(str)
y = df['sentiment_label']

In [10]:
# Step 3: Text Vectorization
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_vec = vectorizer.fit_transform(X)

# Step 4: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

# Step 5: Train a Classifier
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 6: Evaluate
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))


Accuracy: 0.7230769230769231
Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      0.16      0.27        45
     Neutral       0.73      0.87      0.79       120
    Positive       0.70      0.81      0.75        95

    accuracy                           0.72       260
   macro avg       0.81      0.61      0.60       260
weighted avg       0.76      0.72      0.69       260

